# Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pickle as pkl
import xarray

In [3]:
from backend import data_paths
from backend import evaluation_utils
from backend import gauge_groups_utils
from backend import loading_utils
from backend import metrics_utils
from backend import return_period_metrics

In [4]:
RESTART = True

# Load Data

## Experiments

In [5]:
GOOGLE_EXPERIMENTS = ['hydrologically_separated']
GLOFAS_EXPERIMENTS = [metrics_utils.GLOFAS_VARIABLE]

### GloFAS Data

In [6]:
all_gauges = gauge_groups_utils.get_full_gauge_group()
print(f'There are {len(all_gauges)} gauges.')

There are 5678 gauges.


In [ ]:
glofas_model_runs = loading_utils.load_glofas_model_runs(gauges=all_gauges)

## ⚠️ Data Requirements

**This notebook requires data files that are not included in the repository.**

Before running the cells below, you need to:

1. **Download model data from Zenodo:**
   - Visit: https://doi.org/10.5281/zenodo.10397664
   - Download and extract the tarballs
   - Place the extracted folders (`model_data/`, `metadata/`, `metrics/`) in the repository root

2. **Verify the following directories exist:**
   - `model_data/google/dual_lstm/` - Google model outputs
   - `model_data/GRDCstattions_GloFASv40/` - GloFAS model outputs
   - `grdc_data/` - GRDC observation data (optional, for recalculating metrics)
   - `metadata/` - Basin attributes and metadata

**Expected data size:** Several GB (varies by selected data)

**Alternative:** If you only want to recreate figures from the paper, you can skip to the `figure_*.ipynb` notebooks which use pre-calculated metrics from Zenodo.

In [8]:
# Check if required data directories exist
import os
from pathlib import Path

repo_root = Path(__file__).parent.parent if '__file__' in globals() else Path.cwd().parent
required_dirs = {
    'model_data': repo_root / 'model_data',
    'metadata': repo_root / 'metadata',
    'grdc_data': repo_root / 'grdc_data',
    'metrics': repo_root / 'metrics'
}

print("Data Directory Status:")
print("-" * 50)
missing_dirs = []
for name, path in required_dirs.items():
    exists = path.exists()
    status = "✓ EXISTS" if exists else "✗ MISSING"
    print(f"{name:15} {status:10} {path}")
    if not exists:
        missing_dirs.append(name)

print("-" * 50)
if missing_dirs:
    print(f"\n⚠️  Missing {len(missing_dirs)} required directories: {', '.join(missing_dirs)}")
    print("\nPlease download data from: https://doi.org/10.5281/zenodo.10397664")
else:
    print("\n✓ All required directories found! You can proceed with the notebook.")

Data Directory Status:
--------------------------------------------------
model_data      ✗ MISSING  c:\Users\ASUS\OneDrive\SCI\Github\prediction-of-extreme-floods-in-ungauged-watersheds\model_data
metadata        ✗ MISSING  c:\Users\ASUS\OneDrive\SCI\Github\prediction-of-extreme-floods-in-ungauged-watersheds\metadata
grdc_data       ✗ MISSING  c:\Users\ASUS\OneDrive\SCI\Github\prediction-of-extreme-floods-in-ungauged-watersheds\grdc_data
metrics         ✗ MISSING  c:\Users\ASUS\OneDrive\SCI\Github\prediction-of-extreme-floods-in-ungauged-watersheds\metrics
--------------------------------------------------

⚠️  Missing 4 required directories: model_data, metadata, grdc_data, metrics

Please download data from: https://doi.org/10.5281/zenodo.10397664


In [ ]:
glofas_gauges = set(glofas_model_runs.gauge_id.values)

### Google Data

In [ ]:
google_model_runs = loading_utils.load_all_experimental_model_runs(
    gauges=glofas_gauges,
    experiments=GOOGLE_EXPERIMENTS
)

In [ ]:
google_gauges = set(google_model_runs[GOOGLE_EXPERIMENTS[0]].gauge_id.values)

### GRDC Data

In [ ]:
grdc_observation_data = loading_utils.load_grdc_data()

## Overlapping Gauge Groups

In [ ]:
gauges = list(glofas_gauges.intersection(google_gauges))
print(f'There are {len(gauges)} gauges that exist for both models.')

# Time Periods

In [ ]:
google_validation_time_periods = {
    gauge: ['2014-01-01', '2023-01-01'] for gauge in gauges
}

# Google Model Metrics

In [ ]:
# Add observation data to model run xarrays, and delete redundant varaible to save memory.
for experiment in google_model_runs.keys():
    google_model_runs[experiment] = xarray.merge(
        [google_model_runs[experiment], grdc_observation_data])

## Metrics: 2014 - Present

In [ ]:
working_path = data_paths.GOOGLE_2014_RETURN_PERIOD_METRICS_DIR
experiments = GOOGLE_EXPERIMENTS
gauge_list = gauges
ds_dict = google_model_runs
evaluation_time_periods = google_validation_time_periods
lead_times = None

missing_gauges = return_period_metrics.compute_metrics(
    restart=RESTART,
    working_path=working_path,
    experiments=experiments,
    gauge_list=gauge_list,
    sim_variable=metrics_utils.GOOGLE_VARIABLE,
    obs_variable=metrics_utils.OBS_VARIABLE,
    ds_dict=ds_dict,
    evaluation_time_periods=evaluation_time_periods,
    lead_times=lead_times
)

In [ ]:
for experiment in experiments:
    print(f'Experiment {experiment} has {len(missing_gauges[experiment])} missing gauges.')

In [ ]:
metrics = metrics_utils.load_metrics_df(
    filepath=working_path / experiment / 'precision' / f'{gauges[0]}.csv')
metrics

## Metrics: 1980 - Present

In [ ]:
working_path = data_paths.GOOGLE_1980_RETURN_PERIOD_METRICS_DIR
experiments = GOOGLE_EXPERIMENTS
gauge_list = gauges
ds_dict = google_model_runs
evaluation_time_periods = None
lead_times = [0]

missing_gauges = return_period_metrics.compute_metrics(
    restart=RESTART,
    working_path=working_path,
    experiments=experiments,
    gauge_list=gauge_list,
    sim_variable=metrics_utils.GOOGLE_VARIABLE,
    obs_variable=metrics_utils.OBS_VARIABLE,
    ds_dict=ds_dict,
    evaluation_time_periods=evaluation_time_periods,
    lead_times=lead_times
)

In [ ]:
for experiment in experiments:
    print(f'Experiment {experiment} has {len(missing_gauges[experiment])} missing gauges.')

In [ ]:
metrics = metrics_utils.load_metrics_df(
    filepath=working_path / experiment / 'precision' / f'{gauges[0]}.csv')
metrics

## Delete Variables to Clear Memory

In [ ]:
del google_model_runs

# GloFAS

In [ ]:
# Merge everything into one large xarray.
# This xarray merge takes ... forever ...
glofas_model_runs = xarray.merge(
    [glofas_model_runs, grdc_observation_data.sel(lead_time=0)])

## Metrics: 2014 - Present

In [ ]:
working_path = data_paths.GLOFAS_2014_RETURN_PERIOD_METRICS_DIR
experiments = GLOFAS_EXPERIMENTS
gauge_list = gauges
ds_dict = {metrics_utils.GLOFAS_VARIABLE: glofas_model_runs}
evaluation_time_periods = google_validation_time_periods
lead_times = [0]

missing_gauges = return_period_metrics.compute_metrics(
    restart=RESTART,
    working_path=working_path,
    experiments=experiments,
    gauge_list=gauge_list,
    sim_variable=metrics_utils.GLOFAS_VARIABLE,
    obs_variable=metrics_utils.UNNORMALIZED_OBS_VARIABLE,
    ds_dict=ds_dict,
    evaluation_time_periods=evaluation_time_periods,
    lead_times=lead_times
)

In [ ]:
for experiment in experiments:
    print(f'Experiment {experiment} has {len(missing_gauges[experiment])} missing gauges.')

In [ ]:
metrics = metrics_utils.load_metrics_df(
    filepath=working_path / experiment / 'precision' / f'{gauges[0]}.csv')
metrics

## Metrics: 1980 - Present

In [ ]:
working_path = data_paths.GLOFAS_1980_RETURN_PERIOD_METRICS_DIR
experiments = GLOFAS_EXPERIMENTS
gauge_list = gauges
ds_dict = {metrics_utils.GLOFAS_VARIABLE: glofas_model_runs}
evaluation_time_periods = None
lead_times = [0]

missing_gauges = return_period_metrics.compute_metrics(
    restart=RESTART,
    working_path=working_path,
    experiments=experiments,
    gauge_list=gauge_list,
    sim_variable=metrics_utils.GLOFAS_VARIABLE,
    obs_variable=metrics_utils.UNNORMALIZED_OBS_VARIABLE,
    ds_dict=ds_dict,
    evaluation_time_periods=evaluation_time_periods,
    lead_times=lead_times
)

In [ ]:
for experiment in experiments:
    print(f'Experiment {experiment} has {len(missing_gauges[experiment])} missing gauges.')

In [ ]:
metrics = metrics_utils.load_metrics_df(
    filepath=working_path / experiment / 'precision' / f'{gauges[0]}.csv')
metrics

# Collect Return Period Metrics in Pickle Files

In [ ]:
_DATASET_RETURN_PERIOD_METRICS_PATH = {
    'google_2014': data_paths.GOOGLE_2014_RETURN_PERIOD_METRICS_DIR,
    'google_1980': data_paths.GOOGLE_1980_RETURN_PERIOD_METRICS_DIR,
    'glofas_2014': data_paths.GLOFAS_2014_RETURN_PERIOD_METRICS_DIR,
    'glofas_1980': data_paths.GLOFAS_1980_RETURN_PERIOD_METRICS_DIR,
}

In [ ]:
from backend import data_paths

precisions_by_lead_time = {}
recalls_by_lead_time = {}

precisions_by_return_period = {}
recalls_by_return_period = {}

loading_utils.create_remote_folder_if_necessary(data_paths.CONCATENATED_RETURN_PERIOD_DICTS_DIR)

for dataset, data_path in _DATASET_RETURN_PERIOD_METRICS_PATH.items():

    print(f'Working on {dataset} ...')

    file_path = data_paths.CONCATENATED_RETURN_PERIOD_DICTS_DIR / f'{dataset}_return_period_dicts.pkl'

    if 'glofas' in dataset:
        experiments = GLOFAS_EXPERIMENTS
    else:
        experiments = GOOGLE_EXPERIMENTS

    precisions_by_lead_time[dataset] = evaluation_utils.load_return_period_metrics(
        base_path=data_path,
        experiments=experiments,
        gauges=gauges,
        metric='precision'
    )
    recalls_by_lead_time[dataset] = evaluation_utils.load_return_period_metrics(
        base_path=data_path,
        experiments=experiments,
        gauges=gauges,
        metric='recall'
    )

    with open(file_path, 'wb') as f:
        pkl.dump(
            [
                precisions_by_lead_time[dataset],
                recalls_by_lead_time[dataset],
            ], f
        )

    print(f'Finished with {dataset}. \n')